In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import os

data_path = 'hubeau/'
files = [f for f in os.listdir(data_path) if f.endswith('.csv') and 'obstr' in f]
for f in sorted(files):
    print(f)

hubeau_obstr_A161003001_H_20260604_20260528.csv
hubeau_obstr_A214010001_H_20260604_20260505.csv
hubeau_obstr_A214010001_Q_20260604_20260505.csv
hubeau_obstr_A236003001_H_20260604_20260520.csv
hubeau_obstr_A236003001_Q_20260604_20260520.csv
hubeau_obstr_A348020001_H_20260604_20260520.csv
hubeau_obstr_A348020001_Q_20260604_20260520.csv


In [6]:
dfs = {}
for f in sorted(files):
    parts = f.replace('.csv', '').split('_')
    station = parts[2]
    grandeur = parts[3]
    key = f"{station}_{grandeur}"
    df = pd.read_csv(data_path + f)
    df['date_obs'] = pd.to_datetime(df['date_obs'])
    df = df.sort_values('date_obs')
    dfs[key] = df
    print(f"{key} — {len(df)} rows — {df['date_obs'].min().date()} → {df['date_obs'].max().date()}")

A161003001_H — 1000 rows — 2026-05-28 → 2026-06-04
A214010001_H — 328 rows — 2026-05-05 → 2026-06-04
A214010001_Q — 328 rows — 2026-05-05 → 2026-06-04
A236003001_H — 274 rows — 2026-05-20 → 2026-06-04
A236003001_Q — 274 rows — 2026-05-20 → 2026-06-04
A348020001_H — 401 rows — 2026-05-20 → 2026-06-04
A348020001_Q — 401 rows — 2026-05-20 → 2026-06-04


In [7]:
def assign_risk(df, col='resultat_obs'):
    q33 = df[col].quantile(0.33)
    q66 = df[col].quantile(0.66)
    df['risk_level'] = df[col].apply(lambda val: 1 if val <= q33 else (2 if val <= q66 else 3))
    return df

def add_features(df, col='resultat_obs'):
    df = df.sort_values('date_obs')
    df['hour'] = df['date_obs'].dt.hour
    df['dayofweek'] = df['date_obs'].dt.dayofweek
    df['month'] = df['date_obs'].dt.month
    df['rolling_mean_3h'] = df[col].rolling(window=18).mean()
    df['rolling_max_3h'] = df[col].rolling(window=18).max()
    df['rolling_std_3h'] = df[col].rolling(window=18).std()
    df['diff_1'] = df[col].diff(1)
    df['diff_6'] = df[col].diff(6)
    return df.dropna()

for key in ['A161003001_H', 'A214010001_H', 'A236003001_H', 'A348020001_H']:
    dfs[key] = assign_risk(dfs[key])
    dfs[key] = add_features(dfs[key])
    print(f"{key} — {len(dfs[key])} rows")

A161003001_H — 983 rows
A214010001_H — 311 rows
A236003001_H — 257 rows
A348020001_H — 384 rows


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
import xgboost as xgb

features = ['resultat_obs', 'hour', 'dayofweek', 'month', 'rolling_mean_3h', 'rolling_max_3h', 'rolling_std_3h', 'diff_1', 'diff_6']
target = 'risk_level'

df_model = pd.concat([dfs[k] for k in ['A161003001_H', 'A214010001_H', 'A236003001_H', 'A348020001_H']])
X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
}

results = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    elif name == 'XGBoost':
        model.fit(X_train, y_train_xgb)
        y_pred = model.predict(X_test) + 1
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results[name] = f1
    print(f"\n=== {name} — F1: {f1:.3f} ===")
    print(classification_report(y_test, y_pred))


=== Logistic Regression — F1: 0.457 ===
              precision    recall  f1-score   support

           1       0.43      0.51      0.47       136
           2       0.39      0.39      0.39       131
           3       0.59      0.47      0.52       120

    accuracy                           0.45       387
   macro avg       0.47      0.45      0.46       387
weighted avg       0.47      0.45      0.46       387


=== Random Forest — F1: 0.982 ===
              precision    recall  f1-score   support

           1       0.98      0.96      0.97       136
           2       0.98      0.99      0.99       131
           3       0.98      0.99      0.98       120

    accuracy                           0.98       387
   macro avg       0.98      0.98      0.98       387
weighted avg       0.98      0.98      0.98       387


=== XGBoost — F1: 0.974 ===
              precision    recall  f1-score   support

           1       0.98      0.95      0.96       136
           2       0.98 

In [13]:
print("=== BENCHMARK SUMMARY ===\n")
for name, f1 in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:25} — F1: {f1:.3f}")

print(f"\nBest model: Random Forest (F1: {results['Random Forest']:.3f})")

=== BENCHMARK SUMMARY ===

Random Forest             — F1: 0.982
XGBoost                   — F1: 0.974
Logistic Regression       — F1: 0.457

Best model: Random Forest (F1: 0.982)


In [ ]:
print("=== SUMMARY PER STATION ===\n")
for station, group in df_all.groupby('station'):
    print(f"Station: {station}")
    print(f"  Period : {group['date_obs'].min()} → {group['date_obs'].max()}")
    print(f"  Rows   : {len(group)}")
    print(f"  Min    : {group['resultat_obs'].min()} mm")
    print(f"  Max    : {group['resultat_obs'].max()} mm")
    print(f"  Mean   : {group['resultat_obs'].mean():.1f} mm")
    print(f"  Missing: {group['resultat_obs'].isna().sum()}")
    print()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

data_path = 'hubeau/'
files = [f for f in os.listdir(data_path) if f.endswith('.csv') and 'obstr' in f]
print(f"Files found: {len(files)}")
for f in sorted(files):
    print(f" - {f}")

In [ ]:
import pandas as pd
import os

data_path = 'hubeau/'
files = [f for f in os.listdir(data_path) if f.endswith('.csv') and 'obstr' in f]

latest = {}
for f in files:
    parts = f.replace('.csv', '').split('_')
    station = parts[2]
    grandeur = parts[3]
    key = f"{station}_{grandeur}"
    if key not in latest or f > latest[key]:
        latest[key] = f

print(f"Keeping {len(latest)} files:")
for k, v in sorted(latest.items()):
    print(f" - {v}")

for f in files:
    parts = f.replace('.csv', '').split('_')
    station = parts[2]
    grandeur = parts[3]
    key = f"{station}_{grandeur}"
    if latest[key] != f:
        os.remove(data_path + f)
        print(f"Deleted: {f}")

In [ ]:
dfs = {}
for f in sorted(latest.values()):
    parts = f.replace('.csv', '').split('_')
    station = parts[2]
    grandeur = parts[3]
    key = f"{station}_{grandeur}"
    df = pd.read_csv(data_path + f)
    df['date_obs'] = pd.to_datetime(df['date_obs'])
    df = df.sort_values('date_obs', ascending=False)
    dfs[key] = df
    print(f"{key} — {len(df)} rows — {df['date_obs'].min().date()} → {df['date_obs'].max().date()}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

stations_h = ['A161003001_H', 'A214010001_H', 'A348020001_H']
titles = ['Ill à Colmar (A161003001)', 'Fecht à Ostheim (A214010001)', 'Zorn à Waltenheim (A348020001)']

for ax, key, title in zip(axes, stations_h, titles):
    df = dfs[key]
    ax.plot(df['date_obs'], df['resultat_obs'], linewidth=0.8, color='steelblue')
    ax.set_tit
    le(title)
    ax.set_ylabel('Height (mm)')

plt.tight_layout()
plt.savefig('hubeau/eda_water_levels.png', dpi=150)
plt.show()
print("Graph saved.")

In [ ]:
print("=== SUMMARY ===\n")
for key, df in dfs.items():
    print(f"{key}")
    print(f"  Rows    : {len(df)}")
    print(f"  Min     : {df['resultat_obs'].min()}")
    print(f"  Max     : {df['resultat_obs'].max()}")
    print(f"  Mean    : {df['resultat_obs'].mean():.1f}")
    print(f"  Missing : {df['resultat_obs'].isna().sum()}")
    print()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

pairs = [
    ('A214010001_H', 'A214010001_Q', 'Fecht à Ostheim'),
    ('A236003001_H', 'A236003001_Q', 'Ill à Kogenheim'),
]

for i, (h_key, q_key, title) in enumerate(pairs):
    df_h = dfs[h_key]
    df_q = dfs[q_key]
    
    axes[i][0].plot(df_h['date_obs'], df_h['resultat_obs'], color='steelblue', linewidth=0.8)
    axes[i][0].set_title(f"{title} — Height (mm)")
    
    axes[i][1].plot(df_q['date_obs'], df_q['resultat_obs'], color='darkorange', linewidth=0.8)
    axes[i][1].set_title(f"{title} — Flow (L/s)")

plt.tight_layout()
plt.show()

In [ ]:
def assign_risk(df, col='resultat_obs'):
    q33 = df[col].quantile(0.33)
    q66 = df[col].quantile(0.66)
    
    def risk(val):
        if val <= q33:
            return 1
        elif val <= q66:
            return 2
        else:
            return 3
    
    df['risk_level'] = df[col].apply(risk)
    return df, q33, q66

for key in ['A161003001_H', 'A214010001_H', 'A236003001_H', 'A348020001_H']:
    dfs[key], q33, q66 = assign_risk(dfs[key])
    print(f"{key}")
    print(f"  Low  (1) ≤ {q33:.0f} mm : {(dfs[key]['risk_level']==1).sum()} obs")
    print(f"  Med  (2) ≤ {q66:.0f} mm : {(dfs[key]['risk_level']==2).sum()} obs")
    print(f"  High (3) > {q66:.0f} mm : {(dfs[key]['risk_level']==3).sum()} obs")
    print()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
stations_h = ['A161003001_H', 'A214010001_H', 'A236003001_H', 'A348020001_H']
colors = {1: 'green', 2: 'orange', 3: 'red'}

for ax, key in zip(axes.flatten(), stations_h):
    df = dfs[key]
    for level in [1, 2, 3]:
        subset = df[df['risk_level'] == level]
        ax.scatter(subset['date_obs'], subset['resultat_obs'], 
                   c=colors[level], s=1, label=f'Level {level}')
    ax.set_title(key)
    ax.set_ylabel('Height (mm)')
    ax.legend(markerscale=5, fontsize=7)

plt.tight_layout()
plt.savefig('hubeau/eda_risk_levels.png', dpi=150)
plt.show()

In [ ]:
def add_features(df, col='resultat_obs'):
    df = df.sort_values('date_obs')
    df['hour'] = df['date_obs'].dt.hour
    df['dayofweek'] = df['date_obs'].dt.dayofweek
    df['month'] = df['date_obs'].dt.month
    df['rolling_mean_3h'] = df[col].rolling(window=18).mean()
    df['rolling_max_3h'] = df[col].rolling(window=18).max()
    df['rolling_std_3h'] = df[col].rolling(window=18).std()
    df['diff_1'] = df[col].diff(1)
    df['diff_6'] = df[col].diff(6)
    df = df.dropna()
    return df

for key in ['A161003001_H', 'A214010001_H', 'A236003001_H', 'A348020001_H']:
    dfs[key] = add_features(dfs[key])
    print(f"{key} — {len(dfs[key])} rows — {dfs[key].columns.tolist()}")

In [ ]:
import pandas as pd

df_model = pd.concat([dfs[k] for k in ['A161003001_H', 'A214010001_H', 'A236003001_H', 'A348020001_H']])

features = ['resultat_obs', 'hour', 'dayofweek', 'month', 'rolling_mean_3h', 'rolling_max_3h', 'rolling_std_3h', 'diff_1', 'diff_6']
target = 'risk_level'

X = df_model[features]
y = df_model[target]

print(f"Dataset : {X.shape}")
print(f"Target distribution :\n{y.value_counts().sort_index()}")